# 02: Build Vietnamese Bag of Hallucinations

This notebook builds Vietnamese Bag-of-Hallucinations (BoH) artifacts for one or more ASR models.

A BoH artifact is a model-specific list of phrases that an ASR model repeatedly emits when the input is **non-speech audio**. The list is not hand-written; it is constructed empirically by running the model over noise/silence audio and counting repeated non-empty transcripts.

## Why multi-model?

The hallucination pattern is coupled to the ASR model, tokenizer, decoder settings, and quantization/export format. A BoH collected from `PhoWhisper-tiny` should not be blindly reused for `PhoWhisper-base`, `PhoWhisper-small`, or a fine-tuned model.

This notebook therefore uses a model registry:

```text
MODEL_REGISTRY
  phowhisper_tiny   -> huuquyet/PhoWhisper-tiny
  phowhisper_base   -> huuquyet/PhoWhisper-base
  phowhisper_small  -> huuquyet/PhoWhisper-small
  phowhisper_medium -> huuquyet/PhoWhisper-medium
```

You choose which models to run with `SELECTED_MODEL_KEYS`. The default runs `tiny` and `base`; `small` and `medium` are registered but opt-in because they are much heavier.

## Pipeline

```text
Drive noise dataset and Drive model cache
  /datasets/noise_for_boh/manifest.jsonl
  /datasets/noise_for_boh/wav/*.wav
            |
            v
Load manifest + audio files
            |
            v
For each selected ASR model
            |
            v
Download/cache ONNX model under `/content/drive/MyDrive/shrike-7/models/`
            |
            v
Run VietnameseASR over non-speech audio
            |
            v
Raw transcript per noise file
            |
            +-- empty text     -> OK, no hallucination
            +-- non-empty text -> hallucination candidate
                                  |
                                  v
                        normalize transcript
                                  |
                                  v
                        Counter(normalized_text)
                                  |
                                  v
               keep if count >= MIN_COUNT and len >= MIN_CHARS
                                  |
                                  v
              exports/boh/{model_key}_vi_boh_v1.json
```

## Output files

```text
/content/drive/MyDrive/shrike-7/outputs/{RUN_ID}/logs/boh_runs/{model_key}/phowhisper_noise_outputs.jsonl
/content/drive/MyDrive/shrike-7/outputs/{RUN_ID}/boh/{model_key}_vi_boh_v1.json
/content/drive/MyDrive/shrike-7/outputs/{RUN_ID}/boh/boh_build_summary.json
/content/drive/MyDrive/shrike-7/outputs/{RUN_ID}/boh/cross_model_consensus.json
```

The JSONL file is for audit/debug. The BoH JSON file is the runtime artifact candidate. The summary file compares hallucination rates across selected models.


## Research backing and design review

The main reference is **Investigation of Whisper ASR Hallucinations Induced by Non-Speech Audio** (Baranski et al., ICASSP 2025). The paper constructs hallucination lists by running Whisper on non-speech audio, then uses Bag-of-Hallucinations matching as part of an ASR robustness pipeline. It also separates looping failures from phrase-level hallucination matching and shows that the strongest practical pipeline combines VAD, de-looping, and BoH.

Shrike-7 adapts the idea as follows:

- The paper studies Whisper large-v3; Shrike-7 uses PhoWhisper ONNX variants for Vietnamese.
- The BoH must be model-specific because different PhoWhisper variants may hallucinate different phrases.
- This notebook produces candidate artifacts, not final production blocklists.
- Manual review is required before using any phrase list in runtime.
- Runtime matching should later use Aho-Corasick for efficient multi-pattern matching.

Design risks and controls:

| Risk | Why it matters | Control |
|---|---|---|
| False positives | A phrase may be valid user speech in another context | Keep `keep=true` for manual review before runtime use |
| Small collection set | A few hundred noise files are not production-scale | Store `num_noise_samples` and scale the dataset later |
| Evaluation leakage | Evaluating on the same noise used to build BoH overstates performance | Use a held-out noise split for benchmark notebooks |
| Model coupling | BoH depends on model/tokenizer/decoder/export | Build one BoH per model key |
| Normalization mismatch | Same phrase can differ by whitespace/case/punctuation | Normalize before counting and store raw examples |
| Novel hallucinations | BoH only catches phrases already observed | Keep VAD, de-looping, repetition heuristics, and duration checks |

References:

- Baranski et al., ICASSP 2025: https://arxiv.org/abs/2501.11378
- PhoWhisper: https://github.com/VinAIResearch/PhoWhisper
- faster-whisper VAD defaults: https://github.com/SYSTRAN/faster-whisper/blob/master/faster_whisper/vad.py
- Calm-Whisper future work: https://arxiv.org/abs/2505.12969


## 0. Check runtime

This notebook can run on CPU, but the runtime check is useful for documenting the Colab environment.

In [ ]:
!nvidia-smi || true

## 1. Experiment configuration

`MAX_FILES` is useful for a smoke run. Set `MAX_FILES = 20` to validate the pipeline quickly, then set it back to `None` for the full dataset.

`SELECTED_MODEL_KEYS` controls which models are processed. The default is `tiny + base`. Add `phowhisper_small` or `phowhisper_medium` only when you have enough Colab time and Drive space.

In [ ]:
PROJECT = "shrike-7"
RUN_NAME = "d2_5_build_vietnamese_boh"
SEED = 42

# Colab Free default: start with a smoke run. Set to None for the full run.
MAX_FILES = 20
MIN_COUNT = 2
MIN_CHARS = 5
NUM_THREADS = 4

USE_DRIVE_CACHE = True
SYNC_OUTPUTS_TO_DRIVE = True
USE_GPU_IF_AVAILABLE = True
ONNX_PROVIDERS_PRIORITY = ["CUDAExecutionProvider", "CPUExecutionProvider"]

ASR_CONFIG = {
    "language": "vi",
    "task": "transcribe",
    "decode": "greedy",
    "num_beams": 1,
    "temperature": 0.0,
    "max_new_tokens": 128,
}

MAX_RUNTIME_MINUTES = 240
WARN_DISK_USAGE_GB = 30
MAX_DISK_USAGE_GB = 50
MAX_MODEL_PARAMS_M = 250
OVERRIDE_SIZE_LIMIT = False
MIN_PLAUSIBLE_HALLUCINATION_RATE = 0.05
MAX_PLAUSIBLE_HALLUCINATION_RATE = 0.80

CROSS_MODEL_CONSENSUS_CONFIG = {
    "min_models_agreeing": 2,
    "fuzzy_match_threshold": 0.85,
    "use_fuzzy_matching": False,
}

PINNED_VERSIONS_TO_LOG = [
    "numpy",
    "onnxruntime",
    "transformers",
    "huggingface_hub",
    "librosa",
    "soundfile",
    "rich",
]

SELECTED_MODEL_KEYS = [
    "phowhisper_tiny",
    "phowhisper_base",
]

# This model's artifact is also copied to data/asr/vi_boh_v1.json for current runtime compatibility.
RUNTIME_MODEL_KEY = "phowhisper_tiny"

MODEL_REGISTRY = {
    "phowhisper_tiny": {
        "repo_id": "huuquyet/PhoWhisper-tiny",
        "drive_subdir": "phowhisper-tiny-onnx",
        "params_m": 39,
        "description": "Fast default baseline for edge-oriented Shrike-7 runtime.",
    },
    "phowhisper_base": {
        "repo_id": "huuquyet/PhoWhisper-base",
        "drive_subdir": "phowhisper-base-onnx",
        "params_m": 74,
        "description": "More accurate comparison model; still feasible for Colab BoH collection.",
    },
    "phowhisper_small": {
        "repo_id": "huuquyet/PhoWhisper-small",
        "drive_subdir": "phowhisper-small-onnx",
        "params_m": 244,
        "description": "Heavier optional model. Enable only for extended experiments.",
    },
    "phowhisper_medium": {
        "repo_id": "huuquyet/PhoWhisper-medium",
        "drive_subdir": "phowhisper-medium-onnx",
        "params_m": 769,
        "description": "Very heavy optional model. Expect large downloads and long runtime.",
    },
}

MODEL_ALLOW_PATTERNS = [
    "onnx/encoder_model.onnx",
    "onnx/decoder_model.onnx",
    "config.json",
    "generation_config.json",
    "preprocessor_config.json",
    "tokenizer.json",
    "tokenizer_config.json",
    "vocab.json",
    "merges.txt",
    "normalizer.json",
    "added_tokens.json",
    "special_tokens_map.json",
    "quantize_config.json",
]

## 2. Mount Google Drive

Drive stores datasets, model caches, raw logs, and BoH artifacts. These files should not be committed to git.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 3. Declare paths and validate repository state

Notebook 02 can clone the repository into `/content/shrike-7` if it is missing. Notebook 00 is still useful for a clean full setup, but it is no longer mandatory for this notebook. Notebook 01 must still create the noise dataset manifest in Google Drive before this notebook can build BoH artifacts.

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import json
import os
import random
import shutil
import subprocess
import sys

GITHUB_REPO_URL = "https://github.com/finalflash159/shrike-7.git"
RUN_TIMESTAMP = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
RUN_ID = f"{RUN_NAME}_{RUN_TIMESTAMP}"

COLAB_WORKSPACE = Path("/content/shrike-7")
COLAB_MODELS_DIR = COLAB_WORKSPACE / "models"
COLAB_NOISE_DIR = COLAB_WORKSPACE / "data" / "noise_for_boh"
COLAB_OUTPUT_DIR = COLAB_WORKSPACE / "outputs" / RUN_ID

REPO_DIR = COLAB_WORKSPACE
if not REPO_DIR.exists():
    print(f"Repository not found at {REPO_DIR}. Cloning from {GITHUB_REPO_URL}...")
    subprocess.run(["git", "clone", GITHUB_REPO_URL, str(REPO_DIR)], check=True)
else:
    print(f"Repository already exists: {REPO_DIR}")

%cd /content/shrike-7
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

DRIVE_MOUNT_POINT = Path("/content/drive")
DRIVE_PROJECT_ROOT = DRIVE_MOUNT_POINT / "MyDrive" / "shrike-7"
DRIVE_ROOT = DRIVE_PROJECT_ROOT
DRIVE_MODELS_DIR = DRIVE_PROJECT_ROOT / "models"
DRIVE_NOISE_DIR = DRIVE_PROJECT_ROOT / "datasets" / "noise_for_boh"
DRIVE_OUTPUTS_DIR = DRIVE_PROJECT_ROOT / "outputs" / RUN_ID

NOISE_ROOT = DRIVE_NOISE_DIR
NOISE_DIR = NOISE_ROOT / "wav"
MANIFEST = NOISE_ROOT / "manifest.jsonl"
EXPORT_DIR = DRIVE_OUTPUTS_DIR
BOH_EXPORT_DIR = EXPORT_DIR / "boh"
LOG_DIR = DRIVE_OUTPUTS_DIR / "logs"
BOH_LOG_DIR = LOG_DIR / "boh_runs"
MODEL_ROOT = DRIVE_MODELS_DIR if USE_DRIVE_CACHE else COLAB_MODELS_DIR
CONFIG_SNAPSHOT_PATH = EXPORT_DIR / "config_snapshot.json"
ENVIRONMENT_SNAPSHOT_PATH = EXPORT_DIR / "environment.json"

for path in [COLAB_NOISE_DIR, COLAB_OUTPUT_DIR, DRIVE_OUTPUTS_DIR, BOH_EXPORT_DIR, BOH_LOG_DIR, MODEL_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

print(f"Repository: {REPO_DIR}")
print(f"Run ID: {RUN_ID}")
print(f"Noise manifest: {MANIFEST}")
print(f"Model cache root: {MODEL_ROOT}")
print(f"Using Drive model cache: {USE_DRIVE_CACHE}")
print(f"BoH export directory: {BOH_EXPORT_DIR}")
print(f"BoH log directory: {BOH_LOG_DIR}")

## 4. Install notebook dependencies

This cell installs only the dependencies needed for ONNX ASR, audio loading, Hugging Face downloads, and result tables. Training dependencies are intentionally excluded.

In [ ]:
%pip install -q numpy onnxruntime-gpu transformers librosa soundfile rich huggingface-hub

## 5. Reproducibility and environment snapshot

This records the exact config and library versions for the run. The snapshots make the BoH artifact easier to reproduce and review later.

In [ ]:
import importlib.metadata as md
import platform

import numpy as np


def safe_version(package_name: str) -> str:
    try:
        return md.version(package_name)
    except md.PackageNotFoundError:
        return "not-installed"


try:
    import torch

    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
except ImportError:
    torch = None

np.random.seed(SEED)

config_snapshot = {
    "project": PROJECT,
    "run_name": RUN_NAME,
    "run_id": RUN_ID,
    "seed": SEED,
    "max_files": MAX_FILES,
    "min_count": MIN_COUNT,
    "min_chars": MIN_CHARS,
    "num_threads": NUM_THREADS,
    "selected_model_keys": SELECTED_MODEL_KEYS,
    "runtime_model_key": RUNTIME_MODEL_KEY,
    "model_registry": MODEL_REGISTRY,
    "model_allow_patterns": MODEL_ALLOW_PATTERNS,
    "asr_config": ASR_CONFIG,
    "use_drive_cache": USE_DRIVE_CACHE,
    "sync_outputs_to_drive": SYNC_OUTPUTS_TO_DRIVE,
    "use_gpu_if_available": USE_GPU_IF_AVAILABLE,
    "onnx_providers_priority": ONNX_PROVIDERS_PRIORITY,
    "cross_model_consensus_config": CROSS_MODEL_CONSENSUS_CONFIG,
    "safety": {
        "max_runtime_minutes": MAX_RUNTIME_MINUTES,
        "warn_disk_usage_gb": WARN_DISK_USAGE_GB,
        "max_disk_usage_gb": MAX_DISK_USAGE_GB,
        "max_model_params_m": MAX_MODEL_PARAMS_M,
        "override_size_limit": OVERRIDE_SIZE_LIMIT,
    },
}
CONFIG_SNAPSHOT_PATH.write_text(json.dumps(config_snapshot, ensure_ascii=False, indent=2), encoding="utf-8")

environment_snapshot = {
    "python": sys.version,
    "platform": platform.platform(),
    "packages": {name: safe_version(name) for name in PINNED_VERSIONS_TO_LOG},
}
ENVIRONMENT_SNAPSHOT_PATH.write_text(json.dumps(environment_snapshot, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"Config snapshot saved -> {CONFIG_SNAPSHOT_PATH}")
print(f"Environment snapshot saved -> {ENVIRONMENT_SNAPSHOT_PATH}")

## 6. Resolve selected model configs

This converts the human-editable `SELECTED_MODEL_KEYS` list into full model configs with Drive paths and output paths.

In [ ]:
unknown_keys = [key for key in SELECTED_MODEL_KEYS if key not in MODEL_REGISTRY]
if unknown_keys:
    raise ValueError(f"Unknown model keys: {unknown_keys}. Available keys: {list(MODEL_REGISTRY)}")

if RUNTIME_MODEL_KEY not in SELECTED_MODEL_KEYS:
    raise ValueError("RUNTIME_MODEL_KEY must be included in SELECTED_MODEL_KEYS.")

selected_models = []
for model_key in SELECTED_MODEL_KEYS:
    config = dict(MODEL_REGISTRY[model_key])
    params_m = config.get("params_m")
    if params_m is not None and params_m > MAX_MODEL_PARAMS_M and not OVERRIDE_SIZE_LIMIT:
        raise RuntimeError(
            f"Model {model_key} has {params_m}M params, which exceeds MAX_MODEL_PARAMS_M={MAX_MODEL_PARAMS_M}. "
            "Set OVERRIDE_SIZE_LIMIT=True if you intentionally want to run this model."
        )
    config["model_key"] = model_key
    config["model_dir"] = MODEL_ROOT / config["drive_subdir"]
    config["log_dir"] = BOH_LOG_DIR / model_key
    config["raw_output_path"] = config["log_dir"] / "phowhisper_noise_outputs.jsonl"
    config["boh_output_path"] = BOH_EXPORT_DIR / f"{model_key}_vi_boh_v1.json"
    config["log_dir"].mkdir(parents=True, exist_ok=True)
    config["model_dir"].mkdir(parents=True, exist_ok=True)
    selected_models.append(config)

print("Selected models:")
for config in selected_models:
    print(f"- {config['model_key']}: {config['repo_id']} -> {config['model_dir']}")

## 7. Download or reuse ONNX model files

The download is restricted to ONNX and tokenizer/config files. This avoids downloading large PyTorch weights that are not needed by `VietnameseASR`.

When `USE_DRIVE_CACHE = True`, model files are downloaded to `/content/drive/MyDrive/shrike-7/models/`, not to the cloned repo's local `models/` folder.

In [ ]:
from huggingface_hub import snapshot_download


def ensure_model_files(config: dict) -> None:
    required_files = [
        config["model_dir"] / "onnx" / "encoder_model.onnx",
        config["model_dir"] / "onnx" / "decoder_model.onnx",
        config["model_dir"] / "generation_config.json",
    ]
    if all(path.exists() for path in required_files):
        print(f"Model already cached: {config['model_key']} -> {config['model_dir']}")
        return

    print(f"Downloading {config['repo_id']} -> {config['model_dir']}")
    snapshot_download(
        repo_id=config["repo_id"],
        local_dir=str(config["model_dir"]),
        allow_patterns=MODEL_ALLOW_PATTERNS,
    )


for config in selected_models:
    ensure_model_files(config)

## 8. Load the noise manifest

The manifest is produced by notebook 01. Each row points to a non-speech `.wav` file stored in Drive.

In [ ]:
import json

if not MANIFEST.exists():
    raise RuntimeError(
        f"Manifest not found: {MANIFEST}\n"
        "Run notebooks/01_noise_data_collection.ipynb first."
    )

manifest_rows = []
with MANIFEST.open(encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            manifest_rows.append(json.loads(line))

items = []
missing = []
for row in manifest_rows:
    path = NOISE_ROOT / row["path"]
    if path.exists():
        items.append((path, row))
    else:
        missing.append(str(path))

if MAX_FILES is not None:
    items = items[:MAX_FILES]

if not items:
    raise RuntimeError(
        "No audio files were selected for BoH construction. "
        "Check notebook 01 and the noise_for_boh/wav directory in Drive."
    )

print(f"Manifest rows: {len(manifest_rows)}")
print(f"Selected existing audio files: {len(items)}")
print(f"Missing files: {len(missing)}")
if missing[:5]:
    print("Missing examples:")
    for path in missing[:5]:
        print("-", path)

## 9. Define BoH construction helpers

The helper runs a single model over the selected non-speech files, writes a raw JSONL audit log, counts normalized non-empty transcripts, and exports a model-specific BoH candidate JSON.

In [ ]:
from collections import Counter
import re
import shutil
import time
import unicodedata

import librosa
import numpy as np
import soundfile as sf
from rich.console import Console
from rich.progress import track
from rich.table import Table

from shrike7.asr import VietnameseASR

console = Console()

RUN_STARTED_AT = time.perf_counter()


def select_onnx_providers() -> list[str]:
    import onnxruntime as ort

    available = ort.get_available_providers()
    selected = [provider for provider in ONNX_PROVIDERS_PRIORITY if provider in available]
    if not USE_GPU_IF_AVAILABLE:
        selected = [provider for provider in selected if provider == "CPUExecutionProvider"]
    if not selected:
        selected = ["CPUExecutionProvider"]
    print(f"ONNX Runtime providers available: {available}")
    print(f"ONNX Runtime providers selected: {selected}")
    return selected


ONNX_PROVIDERS = select_onnx_providers()


def assert_safety_limits() -> None:
    elapsed_min = (time.perf_counter() - RUN_STARTED_AT) / 60
    if elapsed_min > MAX_RUNTIME_MINUTES:
        raise TimeoutError(f"Run exceeded MAX_RUNTIME_MINUTES={MAX_RUNTIME_MINUTES}.")

    usage = shutil.disk_usage("/content")
    used_gb = (usage.total - usage.free) / (1024**3)
    if used_gb > MAX_DISK_USAGE_GB:
        raise RuntimeError(f"Disk usage {used_gb:.1f}GB exceeded MAX_DISK_USAGE_GB={MAX_DISK_USAGE_GB}GB.")
    if used_gb > WARN_DISK_USAGE_GB:
        print(f"Warning: disk usage is {used_gb:.1f}GB.")


def normalize_transcript(text: str) -> str:
    """Normalize ASR output for counting BoH candidates."""
    text = unicodedata.normalize("NFC", text)
    text = text.strip().lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip(" \t\n\r.,!?;:\"'“”‘’()[]{}")


def build_boh_for_model(config: dict, items: list[tuple[Path, dict]]) -> dict:
    model_key = config["model_key"]
    console.rule(f"Building BoH for {model_key}")
    asr = VietnameseASR(
        model_dir=config["model_dir"],
        num_threads=NUM_THREADS,
        providers=ONNX_PROVIDERS,
    )

    outputs: list[str] = []
    output_examples: dict[str, list[str]] = {}
    records: list[dict] = []

    t0 = time.perf_counter()
    with config["raw_output_path"].open("w", encoding="utf-8") as f:
        for path, row in track(items, description=f"{model_key}: ASR on non-speech"):
            assert_safety_limits()
            record = {
                "model_key": model_key,
                "model_repo": config["repo_id"],
                "path": row["path"],
                "source": row.get("source", ""),
                "label": row.get("label", ""),
                "expected_transcript": row.get("expected_transcript", ""),
            }
            try:
                audio, sr = sf.read(path)
                if audio.ndim > 1:
                    audio = audio.mean(axis=1)
                audio = audio.astype(np.float32)
                if sr != asr.SAMPLING_RATE:
                    audio = librosa.resample(audio, orig_sr=sr, target_sr=asr.SAMPLING_RATE)

                result = asr.transcribe(audio, max_new_tokens=ASR_CONFIG["max_new_tokens"])
                text = result.text.strip()
                normalized_text = normalize_transcript(text)
                record.update(result.to_dict())
                record["text"] = text
                record["normalized_text"] = normalized_text
                record["error"] = ""
                if normalized_text:
                    outputs.append(normalized_text)
                    examples = output_examples.setdefault(normalized_text, [])
                    if text and text not in examples and len(examples) < 5:
                        examples.append(text)
            except Exception as exc:
                record["text"] = ""
                record["normalized_text"] = ""
                record["error"] = repr(exc)

            records.append(record)
            f.write(json.dumps(record, ensure_ascii=False) + "\n")

    elapsed_s = time.perf_counter() - t0
    counter = Counter(outputs)
    top_phrases = counter.most_common(100)
    total_runs = len(records)
    total_hallucinations = len(outputs)
    hallucination_rate = total_hallucinations / max(total_runs, 1)

    table = Table(title=f"Top hallucinations: {model_key}")
    table.add_column("#", justify="right")
    table.add_column("Count", justify="right")
    table.add_column("Phrase")
    for idx, (phrase, count) in enumerate(top_phrases[:20], start=1):
        examples = output_examples.get(phrase, [phrase])
        table.add_row(str(idx), str(count), examples[0][:120])
    console.print(table)

    boh_candidates = [
        {
            "phrase": phrase,
            "count": count,
            "length": len(phrase),
            "keep": True,
            "examples": output_examples.get(phrase, [phrase]),
        }
        for phrase, count in top_phrases
        if count >= MIN_COUNT and len(phrase) >= MIN_CHARS
    ]

    payload = {
        "metadata": {
            "model_key": model_key,
            "model_repo": config["repo_id"],
            "model_dir": str(config["model_dir"]),
            "num_noise_samples": total_runs,
            "num_non_empty_outputs": total_hallucinations,
            "hallucination_rate": hallucination_rate,
            "selection_rule": f"count >= {MIN_COUNT} and len >= {MIN_CHARS} chars before manual review",
            "normalization": "NFC + lowercase + whitespace collapse + boundary punctuation strip",
            "created_by_notebook": "02_build_vietnamese_boh.ipynb",
            "raw_output_jsonl": str(config["raw_output_path"]),
            "max_files": MAX_FILES,
            "elapsed_s": elapsed_s,
        },
        "boh": boh_candidates,
        "all_unique_outputs_with_count": [
            {
                "phrase": phrase,
                "count": count,
                "examples": output_examples.get(phrase, [phrase]),
            }
            for phrase, count in top_phrases
        ],
    }

    config["boh_output_path"].write_text(
        json.dumps(payload, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    console.print(f"Raw ASR outputs saved -> {config['raw_output_path']}")
    console.print(f"BoH candidates saved -> {config['boh_output_path']}")
    console.print(f"Hallucination rate: {hallucination_rate:.2%}")

    return payload

## 10. Build BoH artifacts for selected models

This cell loops over `selected_models`, builds one BoH JSON per model, and writes a summary table for comparison.

In [ ]:
run_payloads = {}
run_summaries = []

for config in selected_models:
    payload = build_boh_for_model(config, items)
    run_payloads[config["model_key"]] = payload
    meta = payload["metadata"]
    run_summaries.append(
        {
            "model_key": meta["model_key"],
            "model_repo": meta["model_repo"],
            "num_noise_samples": meta["num_noise_samples"],
            "num_non_empty_outputs": meta["num_non_empty_outputs"],
            "hallucination_rate": meta["hallucination_rate"],
            "num_boh_candidates": len(payload["boh"]),
            "boh_output_path": str(next(c for c in selected_models if c["model_key"] == meta["model_key"])["boh_output_path"]),
        }
    )

summary_path = BOH_EXPORT_DIR / "boh_build_summary.json"
summary_path.write_text(json.dumps(run_summaries, ensure_ascii=False, indent=2), encoding="utf-8")

summary_table = Table(title="BoH build summary")
summary_table.add_column("Model")
summary_table.add_column("Repo")
summary_table.add_column("Noise", justify="right")
summary_table.add_column("Non-empty", justify="right")
summary_table.add_column("Hallucination rate", justify="right")
summary_table.add_column("BoH candidates", justify="right")
for row in run_summaries:
    summary_table.add_row(
        row["model_key"],
        row["model_repo"],
        str(row["num_noise_samples"]),
        str(row["num_non_empty_outputs"]),
        f"{row['hallucination_rate']:.2%}",
        str(row["num_boh_candidates"]),
    )
console.print(summary_table)
console.print(f"Summary saved -> {summary_path}")

## 11. Build cross-model consensus

Single-model BoH candidates are useful, but cross-model agreement is stronger evidence. This cell groups exact normalized hallucination phrases across all selected models and assigns confidence tiers:

- **Tier S**: seen in every selected model.
- **Tier A**: seen in all but one selected model, or at least three models.
- **Tier B**: seen in at least `min_models_agreeing` models.
- **Tier C**: seen in only one model, usually model-specific and weaker.

The runtime artifact can still use the selected runtime model's BoH, while this consensus file gives us a cleaner research story and a safer list for later manual pruning.

In [ ]:
from collections import defaultdict

TIER_RANK = {"S": 0, "A": 1, "B": 2, "C": 3}


def consensus_tier(n_models: int, total_models: int) -> str:
    if total_models <= 0:
        return "C"
    if n_models == total_models:
        return "S"
    if n_models >= max(3, total_models - 1):
        return "A"
    if n_models >= CROSS_MODEL_CONSENSUS_CONFIG["min_models_agreeing"]:
        return "B"
    return "C"


phrase_to_models = defaultdict(set)
phrase_to_counts = defaultdict(dict)
phrase_to_examples = defaultdict(list)

for model_key, payload in run_payloads.items():
    for item in payload["boh"]:
        phrase = item["phrase"]
        phrase_to_models[phrase].add(model_key)
        phrase_to_counts[phrase][model_key] = item["count"]
        for example in item.get("examples", []):
            if example not in phrase_to_examples[phrase] and len(phrase_to_examples[phrase]) < 8:
                phrase_to_examples[phrase].append(example)

total_models = len(run_payloads)
consensus_items = []
for phrase, model_keys in phrase_to_models.items():
    n_models = len(model_keys)
    tier = consensus_tier(n_models, total_models)
    consensus_items.append(
        {
            "phrase": phrase,
            "n_models": n_models,
            "models": sorted(model_keys),
            "tier": tier,
            "counts_by_model": phrase_to_counts[phrase],
            "examples": phrase_to_examples[phrase],
            "keep": n_models >= CROSS_MODEL_CONSENSUS_CONFIG["min_models_agreeing"],
        }
    )

consensus_items.sort(key=lambda item: (TIER_RANK[item["tier"]], -item["n_models"], item["phrase"]))
high_confidence_items = [item for item in consensus_items if item["keep"] and item["tier"] in {"S", "A", "B"}]

consensus_payload = {
    "metadata": {
        "project": PROJECT,
        "run_id": RUN_ID,
        "selected_model_keys": sorted(run_payloads.keys()),
        "runtime_model_key": RUNTIME_MODEL_KEY,
        "min_models_agreeing": CROSS_MODEL_CONSENSUS_CONFIG["min_models_agreeing"],
        "use_fuzzy_matching": CROSS_MODEL_CONSENSUS_CONFIG["use_fuzzy_matching"],
        "fuzzy_match_threshold": CROSS_MODEL_CONSENSUS_CONFIG["fuzzy_match_threshold"],
        "note": "Exact normalized phrase consensus. Fuzzy matching is configured but intentionally disabled for the first reproducible version.",
    },
    "consensus": consensus_items,
    "high_confidence": high_confidence_items,
}

consensus_path = BOH_EXPORT_DIR / "cross_model_consensus.json"
consensus_path.write_text(json.dumps(consensus_payload, ensure_ascii=False, indent=2), encoding="utf-8")

consensus_table = Table(title="Cross-model BoH consensus")
consensus_table.add_column("Tier")
consensus_table.add_column("Models", justify="right")
consensus_table.add_column("Phrase")
consensus_table.add_column("Model keys")
for item in consensus_items[:20]:
    phrase = item["phrase"] if len(item["phrase"]) <= 70 else item["phrase"][:67] + "..."
    consensus_table.add_row(
        item["tier"],
        str(item["n_models"]),
        phrase,
        ", ".join(item["models"]),
    )

console.print(consensus_table)
console.print(f"Consensus phrases: {len(consensus_items)}")
console.print(f"High-confidence phrases: {len(high_confidence_items)}")
console.print(f"Consensus saved -> {consensus_path}")


## 12. Copy runtime artifacts into the Colab repo

All model-specific BoH files are copied to `data/asr/boh/`. The `RUNTIME_MODEL_KEY` artifact is also copied to `data/asr/vi_boh_v1.json` for current runtime compatibility. The `data/` directory is ignored by git.

In [ ]:
import shutil

local_boh_dir = REPO_DIR / "data" / "asr" / "boh"
local_boh_dir.mkdir(parents=True, exist_ok=True)

for config in selected_models:
    local_path = local_boh_dir / config["boh_output_path"].name
    shutil.copy2(config["boh_output_path"], local_path)
    print(f"Copied model-specific BoH -> {local_path}")

runtime_config = next(config for config in selected_models if config["model_key"] == RUNTIME_MODEL_KEY)
runtime_boh_path = REPO_DIR / "data" / "asr" / "vi_boh_v1.json"
shutil.copy2(runtime_config["boh_output_path"], runtime_boh_path)
print(f"Copied runtime-compatible BoH for {RUNTIME_MODEL_KEY} -> {runtime_boh_path}")
print("Note: data/ is gitignored; these local copies are for runtime loading only.")

## Manual review checklist

After the notebook finishes, open each `exports/boh/{model_key}_vi_boh_v1.json` file in Google Drive and review the `boh` field:

- Keep phrases that are clearly hallucinated on non-speech audio.
- Remove short or ambiguous phrases that may be valid user speech.
- If there are too many false positives, raise `MIN_COUNT` from `2` to `3` or apply stricter manual review.
- Review `cross_model_consensus.json`; Tier S/A/B phrases are safer candidates for a future manually curated runtime BoH.
- Do not evaluate robustness on the exact same noise files used to build the BoH artifact.
- Record top phrases and hallucination rates in benchmark notes only after the run completes.
